# Mistral

In [1]:
##Removing annoying warnings
import warnings

#IProgress
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    message=".*IProgress not found.*"
)

#Suppress the torch/cuda AMD SMI warning
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    message=".*Can't initialize amdsmi.*"
)
warnings.filterwarnings("ignore")

In [2]:
import os
import torch
import gc
import csv
import time
import pandas as pd

from datasets import load_dataset
from lightning import Trainer
from lightning.pytorch import LightningDataModule, LightningModule
from lightning.pytorch.callbacks import TQDMProgressBar, ModelCheckpoint
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, PeftModel

os.environ["TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL"] = "0"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

In [3]:
model_name = "mistralai/Mistral-7B-Instruct-v0.3"

LORA_DIR = "./lora-fine-tuned-model"
os.makedirs(LORA_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


# Data

In [4]:
#Dataset mode
#0 = 1% dataset for testing
#1 = full dataset
DATASET_TYPE = 1

In [5]:
def load_base_model():
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quant_config,
        device_map="auto",
        trust_remote_code=True,
    )

    return model

def attach_lora(model):
    lora_config = LoraConfig(
        r=8,
        lora_alpha=32,
        lora_dropout=0.1,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj"
        ],
        task_type="CAUSAL_LM"
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model

class DataModule(LightningDataModule):
    def __init__(self, batch_size=2, max_length=128):
        super().__init__()
        self.batch_size = batch_size
        self.max_length = max_length
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

    def setup(self, stage=None):
        #1%
        if DATASET_TYPE == 0:
            dataset = load_dataset(
                "Despina/project_gutenberg",
                "fiction_books",
                split="train",
                streaming=True
            ).shuffle(seed=42, buffer_size=1000)
            self.train_dataset = dataset.take(500)

        #full dataset
        elif DATASET_TYPE == 1:
            dataset = load_dataset(
                "Despina/project_gutenberg",
                "fiction_books",
                split="train",
                streaming=True
            ).shuffle(seed=42, buffer_size=1000)
            self.train_dataset = dataset

    def collate_fn(self, batch):
        texts = [x["text"] for x in batch]

        enc = self.tokenizer(
            texts,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        return enc["input_ids"], enc["input_ids"], enc["attention_mask"]

    def train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            collate_fn=self.collate_fn
        )
    
class QLoRAModule(LightningModule):
    def __init__(self):
        super().__init__()
        base = load_base_model()

        for param in base.parameters():
            param.requires_grad = False

        self.model = attach_lora(base)

    def forward(self, input_ids, labels=None, attention_mask=None):
        return self.model(
            input_ids=input_ids,
            labels=labels,
            attention_mask=attention_mask
        )

    def training_step(self, batch, batch_idx):
        input_ids, labels, attention_mask = batch
        outputs = self(input_ids, labels, attention_mask)
        loss = outputs.loss
        self.log("train_loss", loss)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=2e-4)

# Finetuner

In [ ]:
def train_model(epochs=1, batch_size=2):

    #Load data
    data = DataModule(batch_size=batch_size)
    model = QLoRAModule()

    progress_bar = TQDMProgressBar(refresh_rate=10)

    #Add callback and saving every 100 steps
    checkpoint_callback = ModelCheckpoint(
        dirpath=LORA_DIR,
        filename="checkpoint-{step}",
        save_top_k=3,
        every_n_train_steps=100,
        save_last=True
    )

    #check if checkpoint exist
    resume_checkpoint = None

    if os.path.exists(LORA_DIR):
        checkpoints = [
            os.path.join(LORA_DIR, f)
            for f in os.listdir(LORA_DIR)
            if f.endswith(".ckpt")]

        if checkpoints:
            resume_checkpoint = max(checkpoints, key=os.path.getmtime)
            print("Resuming from checkpoint:", resume_checkpoint)
        else:
            print("No checkpoint found. Starting new.")

    #Train param
    trainer = Trainer(
        max_epochs=epochs,
        accelerator="auto",
        logger=False,
        callbacks=[progress_bar, checkpoint_callback]
    )

    trainer.fit(model, data, ckpt_path=resume_checkpoint)

    #SAVE ONLY LORA ADAPTER
    model.model.save_pretrained(LORA_DIR)
    print("LoRA adapter saved to:", LORA_DIR)

    del model
    gc.collect()
    torch.cuda.empty_cache()

In [7]:
torch.set_float32_matmul_precision('high')
train_model(epochs=1)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

trainable params: 20,971,520 || all params: 7,268,995,072 || trainable%: 0.2885
No checkpoint found. Starting new.


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('AMD Radeon RX 7800 XT') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


README.md: 0.00B [00:00, ?B/s]

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type                 ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ PeftModelForCausalLM │  3.8 B │ train │     0 │
└───┴───────┴──────────────────────┴────────┴───────┴───────┘

Trainable params: 21.0 M                                                                                           
Non-trainable params: 3.8 B                                                                                        
Total params: 3.8 B                                                                                                
Total estimated model params size (MB): 15.1 K                                                                     
Modules in train mode: 2242                                                                                        
Modules in eval mode: 423                                                                                          
Total FLOPs: 0

Training: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

# Generation

In [ ]:
#Load inference and check if model is already in memory
#This way we don't have to realod the weights 15 million times
def setup_inference(use_lora=False):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    base_model = load_base_model()

    if use_lora and os.path.exists(LORA_DIR):
        print("Loading LoRA adapter")
        model = PeftModel.from_pretrained(base_model, LORA_DIR)
    else:
        print("Using base model (no LoRA)")
        model = base_model

    model.eval()
    return model, tokenizer


### Generate prompts

In [ ]:
EN_OUTPUT_DIR = "."

def generate_dataset(use_lora=False, total_generations=20, max_tokens=256,
    prompts_info = [{"genres": "Fantasy, slice of life",
    "prompt": "Write a short story about halloween in Japan"}]):

    print("\n==============================")
    print("Loading model")
    print("Use LoRA:", use_lora)
    print("==============================")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    base_model = load_base_model()

    if use_lora and os.path.exists(LORA_DIR):
        model = PeftModel.from_pretrained(base_model, LORA_DIR)
        print("LoRA adapter attached.")
    else:
        model = base_model
        print("Using base model only.")

    model.eval()

    #Decide file path
    if use_lora:
        save_path = os.path.join(EN_OUTPUT_DIR, "generated_stories.csv")
    else:
        save_path = os.path.join(EN_OUTPUT_DIR, "generated_before.csv")

    #Create CSV with header
    with open(save_path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=["genres", "story"])
        writer.writeheader()

    #Generate stories one by one
    for i in range(total_generations):
        info = prompts_info[i % len(prompts_info)]
        formatted = f"<s>[INST] {info['prompt']} [/INST]"

        inputs = tokenizer(formatted, return_tensors="pt").to(model.device)

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                min_new_tokens=max_tokens // 2,
                do_sample=True,
                temperature=0.8,
                top_p=0.95,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id,
            )

        full_text = tokenizer.decode(output[0], skip_special_tokens=True)
        story_content = full_text.replace(formatted, "").strip()

        #Append to CSV
        with open(save_path, "a", newline="", encoding="utf-8-sig") as f:
            writer = csv.DictWriter(f, fieldnames=["genres", "story"])
            writer.writerow({
                "genres": info["genres"],
                "story": story_content
            })

        print(f"Generated {i + 1}/{total_generations}: {info['genres']}")

    print(f"\nSaved all stories to {save_path}")
    return model, tokenizer

In [ ]:
def full_cleanup(model, tokenizer=None):
    if model is not None:
        del model
    if tokenizer is not None:
        del tokenizer
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
'''
#prompt sample

    prompts_info = [
        {
            "genres": "Fantasy, Adventure",
            "prompt": "Write a fantasy story about a young hero going on a journey."
        },
        {
            "genres": "Romance, Slice of Life",
            "prompt": "Write a slice-of-life romance story about two students."
        }
    ]
'''

#BEFORE fine-tuning
#model, tokenizer = generate_dataset(use_lora=False)
#full_cleanup(model, tokenizer)  #Clean unload

In [ ]:
#AFTER fine-tuning
model, tokenizer = generate_dataset(use_lora=True)
full_cleanup(model, tokenizer) #Clean unload